# Local smoke test — CBIS-DDSM + DeepProbLog

This notebook is a **small end-to-end test** you can run on a laptop. It does not need Kaggle and it does not need the JPEG image pack.

**What it checks**
1. The four CBIS CSV files load and merge.
2. Patients that appear in both official train and test after the merge are moved to train only (no leakage).
3. Concept labels are built from the CSV descriptors.
4. A tiny DeepProbLog model trains on a few dozen examples.
5. Train / validation / test images are **drawn on screen** so you can see what the model is looking at.
6. After training, the **test images** are shown again with the predicted `P(malignant)`.

**What the images are**
There are usually no mammogram JPEGs in `data/`. For this smoke test the notebook builds **concept-coded placeholder images**: three (or four) gray bands that encode shape, margin, calc type, and lesion kind. That is enough to test that DeepProbLog is wired correctly. If a `jpeg/` folder exists next to the CSVs, a later cell also tries to show the real files.

Run from the repo root, or from `notebooks/`, with the project virtualenv kernel.


## Cell: imports

**What:** load the Python libraries used in every later cell.

**Why:** pandas reads the CBIS CSVs, scikit-learn splits by patient, matplotlib draws the image grids, and IPython `display` shows tables in the notebook.


In [ ]:
import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit

print("python:", sys.version.split()[0])
print("pandas:", pd.__version__)


## Cell: smoke-test settings

**What:** one place to change how big and how long this test is.

**Why:** a smoke test should stay small. `SMOKE_N_PATIENTS` and `SMOKE_N_SAMPLES` keep the run on a laptop. Set `RUN_DEEPPROBLOG = False` if you only want to check the CSV split. Set `SHOW_N_IMAGES` to how many pictures to draw in each grid.


In [ ]:
# ---- smoke-test settings (safe to edit) ----
SMOKE_N_PATIENTS = 80   # patients kept in the sampled official train / test
SMOKE_N_SAMPLES = 64    # max rows used for DeepProbLog tensors
VAL_SIZE = 0.2          # fraction of sampled train patients held out as validation
RANDOM_SEED = 42
EPOCHS = 6              # tiny run; this is a wiring test, not a full experiment
BATCH_SIZE = 2
IMAGE_SIZE = 32         # placeholder images are small on purpose
LR = 1e-3
RUN_DEEPPROBLOG = True  # False = stop after data prep
SHOW_N_IMAGES = 8       # how many pictures to draw per split

ROOT = Path.cwd()
if (ROOT / "data").is_dir():
    DATA_DIR = ROOT / "data"
    OUT_DIR = ROOT / "notebooks" / "processed_local"
    PROGRAM_PATH = ROOT / "notebooks" / "programs" / "breast_diagnosis.pl"
elif (ROOT / "../data").is_dir():
    DATA_DIR = (ROOT / "../data").resolve()
    OUT_DIR = (ROOT / "processed_local").resolve()
    PROGRAM_PATH = (ROOT / "programs" / "breast_diagnosis.pl").resolve()
else:
    raise FileNotFoundError("Cannot find data/ folder. Run from repo root or notebooks/.")

OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print("PROGRAM_PATH:", PROGRAM_PATH)


## Cell: load CSVs, merge mass+calc, fix leakage, split

**What:**
- Read the four official CBIS description files from `data/`.
- Tag each row as `mass` or `calcification`.
- Merge mass and calc inside official train, and again inside official test.
- Some patients appear in **both** official splits after that merge (mass in train, calc in test). Those patients are assigned entirely to train and removed from test.
- Sample a small patient set, then cut a **patient-disjoint** validation split from train.

**Why:** DeepProbLog training must not see the same patient in train and test. The assertions at the end fail the cell if leakage is still present.


In [ ]:
CSV_FILENAMES = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test": "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test": "calc_case_description_test_set.csv",
}


def normalize_columns(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    if "breast density" in out.columns:
        out = out.rename(columns={"breast density": "breast_density"})
    return out


def add_meta(df, kind, official_split):
    out = df.copy()
    out["abnormality_kind"] = kind
    out["official_split"] = official_split
    return out


def sample_patients(df, n_patients, seed):
    patients = df["patient_id"].dropna().unique()
    if len(patients) <= n_patients:
        return df.copy()
    chosen = set(random.Random(seed).sample(list(patients), n_patients))
    return df[df["patient_id"].isin(chosen)].copy()


def resolve_cross_split_patient_leakage(train_df, test_df):
    overlap = set(train_df["patient_id"].dropna()) & set(test_df["patient_id"].dropna())
    if not overlap:
        print("No cross-split patient overlap.")
        return train_df, test_df
    print(f"Found {len(overlap)} patients in BOTH official train and test after merge.")
    print("Assigning all their rows to TRAIN and removing them from TEST.")
    leaked = test_df[test_df["patient_id"].isin(overlap)].copy()
    train_fixed = pd.concat([train_df, leaked], ignore_index=True).drop_duplicates().reset_index(drop=True)
    test_fixed = test_df[~test_df["patient_id"].isin(overlap)].reset_index(drop=True)
    return train_fixed, test_fixed


raw = {k: normalize_columns(pd.read_csv(DATA_DIR / fn)) for k, fn in CSV_FILENAMES.items()}
df_train_all = pd.concat(
    [add_meta(raw["mass_train"], "mass", "train"), add_meta(raw["calc_train"], "calcification", "train")],
    ignore_index=True,
)
df_test_all = pd.concat(
    [add_meta(raw["mass_test"], "mass", "test"), add_meta(raw["calc_test"], "calcification", "test")],
    ignore_index=True,
)
print("Before fix train:", len(df_train_all), "| test:", len(df_test_all))
print("Overlap before fix:", len(set(df_train_all.patient_id) & set(df_test_all.patient_id)))

df_train_all, df_test_all = resolve_cross_split_patient_leakage(df_train_all, df_test_all)
print("After fix train:", len(df_train_all), "| test:", len(df_test_all))
print("Overlap after fix:", len(set(df_train_all.patient_id) & set(df_test_all.patient_id)))

df_train_all = sample_patients(df_train_all, SMOKE_N_PATIENTS, RANDOM_SEED)
df_test_all = sample_patients(df_test_all, SMOKE_N_PATIENTS, RANDOM_SEED + 1)

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
tr_idx, va_idx = next(gss.split(df_train_all, groups=df_train_all["patient_id"]))
df_train = df_train_all.iloc[tr_idx].reset_index(drop=True)
df_val = df_train_all.iloc[va_idx].reset_index(drop=True)
df_test = df_test_all.reset_index(drop=True)

tr_p, va_p, te_p = set(df_train.patient_id), set(df_val.patient_id), set(df_test.patient_id)
assert tr_p.isdisjoint(va_p) and tr_p.isdisjoint(te_p) and va_p.isdisjoint(te_p)
print("Leakage checks PASSED")
for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
    print(f"{name:5s}: {len(df):>4} rows | {df['patient_id'].nunique():>3} patients")


## Cell: preview the tables

**What:** show a few rows from train and the pathology mix in each split.

**Why:** before training, confirm that both mass and calcification rows exist, and that malignant is not the only label.


In [ ]:
preview_cols = [
    c for c in [
        "patient_id", "abnormality_kind", "pathology",
        "mass shape", "mass margins", "calc type",
        "left or right breast", "image view",
    ] if c in df_train.columns
]
print("Train preview")
display(df_train[preview_cols].head())
print("Pathology counts")
for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
    print(f"\n{name}")
    display(df["pathology"].value_counts().to_frame("n"))


## Cell: save the smoke-test splits

**What:** write `cbis_train.csv`, `cbis_val.csv`, `cbis_test.csv`, and a small `metadata.json`.

**Why:** so you can inspect the exact rows this run used without re-executing the split.


In [ ]:
df_train.to_csv(OUT_DIR / "cbis_train.csv", index=False)
df_val.to_csv(OUT_DIR / "cbis_val.csv", index=False)
df_test.to_csv(OUT_DIR / "cbis_test.csv", index=False)
meta = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "mode": "local_smoke",
    "counts": {
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
        "train_patients": df_train["patient_id"].nunique(),
        "val_patients": df_val["patient_id"].nunique(),
        "test_patients": df_test["patient_id"].nunique(),
    },
}
(OUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))
print("Saved splits to", OUT_DIR)


## Cell: map CSV descriptors to DeepProbLog concepts

**What:** turn free-text CBIS fields into the symbols the Prolog program uses:
- `shape`: regular / irregular / unknown
- `margin`: circumscribed / spiculated / other / unknown
- `calc`: absent / typically_benign / suspicious
- `kind`: mass / calcification
- `malignant`: 1.0 if pathology is MALIGNANT, else 0.0

**Why:** DeepProbLog does not read the raw CSV strings. The neural nets predict these concepts, and the `.pl` rules combine them into `malignant(X)`.

Any calcification is **not** treated as malignant. Only suspicious types (pleomorphic, amorphous, fine linear branching) map to `suspicious`.


In [ ]:
def tokenize(value):
    if pd.isna(value):
        return []
    text = str(value).upper().replace("-", " ").replace("/", " ").replace("_", " ")
    return [t for t in text.split() if t]


def map_shape(v):
    t = tokenize(v)
    if any(x in {"IRREGULAR", "ARCHITECTURAL", "DISTORTION"} for x in t):
        return "irregular"
    if any(x in {"ROUND", "OVAL", "LOBULATED"} for x in t):
        return "regular"
    return "unknown"


def map_margin(v):
    t = tokenize(v)
    if "SPICULATED" in t:
        return "spiculated"
    if "CIRCUMSCRIBED" in t:
        return "circumscribed"
    return "other" if t else "unknown"


def map_calc(row):
    if str(row.get("abnormality_kind", "")).lower() != "calcification":
        return "absent"
    t = tokenize(row.get("calc type"))
    if any(x in {"PLEOMORPHIC", "AMORPHOUS"} for x in t) or (
        "FINE" in t and "LINEAR" in t
    ) or "BRANCHING" in t:
        return "suspicious"
    return "typically_benign"


def add_concepts(df):
    out = df.copy()
    out["shape_concept"] = out["mass shape"].map(map_shape) if "mass shape" in out.columns else "unknown"
    out["margin_concept"] = out["mass margins"].map(map_margin) if "mass margins" in out.columns else "unknown"
    out["calc_concept"] = out.apply(map_calc, axis=1)
    out["kind_concept"] = out["abnormality_kind"].map(
        lambda x: "calcification" if str(x).strip().lower() == "calcification" else "mass"
    )
    out["malignant"] = out["pathology"].map(lambda x: 1.0 if str(x).upper() == "MALIGNANT" else 0.0)
    return out


df_train = add_concepts(df_train).head(SMOKE_N_SAMPLES).reset_index(drop=True)
df_val = add_concepts(df_val).head(SMOKE_N_SAMPLES).reset_index(drop=True)
df_test = add_concepts(df_test).head(SMOKE_N_SAMPLES).reset_index(drop=True)

concept_cols = ["patient_id", "abnormality_kind", "pathology", "shape_concept", "margin_concept", "calc_concept", "kind_concept", "malignant"]
print("Train concepts")
display(df_train[concept_cols].head(10))
print("calc_concept counts:\n", df_train["calc_concept"].value_counts())
print("kind_concept counts:\n", df_train["kind_concept"].value_counts())


## Cell: build placeholder images and show train / val / test

**What:** each row becomes a small gray image with four horizontal bands:

| band (top → bottom) | encodes |
|---|---|
| 1 | mass shape |
| 2 | mass margin |
| 3 | calc type |
| 4 | lesion kind (mass vs calcification) |

Brighter band = more suspicious / calcification. Then matplotlib draws a grid for **train**, **validation**, and **test**.

**Why:** locally there is usually no JPEG pack. These placeholders give the CNNs a learnable signal so the smoke test can fail or pass for wiring reasons, not because every image is random noise. The titles on each picture are `patient_id`, pathology, and kind.


In [ ]:
import torch

torch.manual_seed(RANDOM_SEED)


def concept_tensor(shape_c, margin_c, calc_c, kind_c, size):
    t = torch.zeros(1, size, size)
    q = max(size // 4, 1)
    t[0, :q, :] = {"irregular": 1.0, "regular": 0.55, "unknown": 0.15}[shape_c]
    t[0, q:2 * q, :] = {"spiculated": 1.0, "circumscribed": 0.55, "other": 0.35, "unknown": 0.15}[margin_c]
    t[0, 2 * q:3 * q, :] = {"suspicious": 1.0, "typically_benign": 0.55, "absent": 0.15}[calc_c]
    t[0, 3 * q:, :] = {"calcification": 0.9, "mass": 0.2}[kind_c]
    return t


def make_tensors(df):
    return [
        concept_tensor(r["shape_concept"], r["margin_concept"], r["calc_concept"], r["kind_concept"], IMAGE_SIZE)
        for _, r in df.iterrows()
    ]


train_tensors = make_tensors(df_train)
val_tensors = make_tensors(df_val)
test_tensors = make_tensors(df_test)


def show_image_grid(title, df, tensors, n=SHOW_N_IMAGES):
    n = min(int(n), len(df), len(tensors))
    if n == 0:
        print(title, ": no images")
        return
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 2.8 * rows))
    axes = np.atleast_1d(axes).ravel()
    for i in range(n):
        ax = axes[i]
        img = tensors[i].squeeze().detach().cpu().numpy()
        ax.imshow(img, cmap="gray", vmin=0, vmax=1)
        row = df.iloc[i]
        ax.set_title(
            f"{row['patient_id']}\n{row['pathology']}\n{row['abnormality_kind']}",
            fontsize=8,
        )
        ax.axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")
    fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    plt.show()


print("These are concept-coded placeholders used by the smoke test (not the original mammograms).")
show_image_grid("Train images (placeholders)", df_train, train_tensors)
show_image_grid("Validation images (placeholders)", df_val, val_tensors)
show_image_grid("Test images (placeholders)", df_test, test_tensors)


## Cell: optional real JPEG preview

**What:** if a `jpeg/` folder exists under the data root, try to open a few test-row files and show them.

**Why:** this smoke test is meant to run without downloading the image pack. When the files *are* present, this cell lets you compare a real mammogram crop to the placeholder grid above. If no JPEG folder is found, it prints that and continues.


In [ ]:
from PIL import Image


def find_local_jpeg_root():
    for cand in (DATA_DIR / "jpeg", DATA_DIR.parent / "jpeg"):
        if cand.is_dir():
            return cand
    return None


JPEG_ROOT = find_local_jpeg_root()
print("Local JPEG root:", JPEG_ROOT)


def try_preview_real_jpegs(df, n=4):
    if JPEG_ROOT is None:
        print("No jpeg/ folder next to the CSVs. Skip real-image preview.")
        return
    shown = 0
    fig, axes = plt.subplots(1, n, figsize=(12, 3))
    axes = np.atleast_1d(axes).ravel()
    for _, row in df.iterrows():
        if shown >= n:
            break
        rel = str(row.get("cropped image file path") or row.get("image file path") or "")
        rel = rel.strip().split("\n")[0].replace(".dcm", "")
        parts = [p for p in rel.replace("\\", "/").split("/") if p]
        folder = None
        for key in reversed(parts):
            key = key[:-4] if key.lower().endswith(".dcm") else key
            hit = JPEG_ROOT / key
            if hit.is_dir():
                folder = hit
                break
        if folder is None:
            continue
        jpgs = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
        if not jpgs:
            continue
        with Image.open(jpgs[0]) as im:
            axes[shown].imshow(im.convert("L"), cmap="gray")
        axes[shown].set_title(f"{row['patient_id']}\n{row['pathology']}", fontsize=8)
        axes[shown].axis("off")
        shown += 1
    for j in range(shown, n):
        axes[j].axis("off")
    fig.suptitle(f"Real JPEG preview ({shown} test images found)")
    fig.tight_layout()
    plt.show()
    if shown == 0:
        print("JPEG folder exists, but none of the sampled CSV paths matched a file.")


try_preview_real_jpegs(df_test)


## Cell: train DeepProbLog

**What:** define tiny CNNs for kind / shape / margin / calc, write the Prolog program, train for a few epochs on the placeholder images.

**Why:** this is the wiring test. The program is the same idea as the full pipeline: mass evidence from irregular or spiculated, calc evidence from suspicious calc **only if** the lesion is classified as a calcification.

If `RUN_DEEPPROBLOG` is False, this cell skips training.


In [ ]:
if not RUN_DEEPPROBLOG:
    print("Skipping DeepProbLog (RUN_DEEPPROBLOG=False). Data-prep smoke OK.")
else:
    import torch.nn as nn
    from problog.logic import Term, Constant
    from deepproblog.dataset import Dataset, DataLoader, QueryDataset
    from deepproblog.query import Query
    from deepproblog.engines import ExactEngine
    from deepproblog.model import Model
    from deepproblog.network import Network
    from deepproblog.train import train_model

    class ConceptCNN(nn.Module):
        def __init__(self, n):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2),
                nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(True),
                nn.AdaptiveAvgPool2d(4),
            )
            self.fc = nn.Linear(32 * 4 * 4, n)
            self.sm = nn.Softmax(dim=1)

        def forward(self, x):
            if not torch.is_tensor(x):
                x = torch.as_tensor(x)
            x = x.to(dtype=torch.float32)
            if x.dim() == 3:
                x = x.unsqueeze(0)
            return self.sm(self.fc(self.net(x).flatten(1)))

    class TensorSource:
        def __init__(self, tensors):
            self.tensors = tensors
        def __getitem__(self, item):
            return self.tensors[int(item[0])]

    class BreastQueryDataset(Dataset):
        def __init__(self, split_name, df):
            self.split_name, self.df = split_name, df.reset_index(drop=True)
        def __len__(self):
            return len(self.df)
        def _t(self, i):
            return Term("tensor", Term(self.split_name, Constant(i)))
        def to_query(self, i):
            return Query(Term("malignant", self._t(i)), p=float(self.df.iloc[i]["malignant"]))
        def concept_queries(self):
            qs = []
            for i, row in self.df.iterrows():
                img = self._t(i)
                qs += [
                    Query(Term("lesion_kind", img, Term(row["kind_concept"])), p=1.0),
                    Query(Term("shape", img, Term(row["shape_concept"])), p=1.0),
                    Query(Term("margin", img, Term(row["margin_concept"])), p=1.0),
                    Query(Term("calc", img, Term(row["calc_concept"])), p=1.0),
                ]
            return qs

    shape_net, margin_net = ConceptCNN(3), ConceptCNN(4)
    calc_net, kind_net = ConceptCNN(3), ConceptCNN(2)
    train_set = BreastQueryDataset("train", df_train)
    val_set = BreastQueryDataset("val", df_val)
    test_set = BreastQueryDataset("test", df_test)
    train_queries = QueryDataset(train_set.to_queries() + train_set.concept_queries())

    PROGRAM = '''
nn(kind_net, [X], Y, [mass, calcification]) :: lesion_kind(X, Y).
nn(shape_net, [X], Y, [regular, irregular, unknown]) :: shape(X, Y).
nn(margin_net, [X], Y, [circumscribed, spiculated, other, unknown]) :: margin(X, Y).
nn(calc_net, [X], Y, [absent, typically_benign, suspicious]) :: calc(X, Y).
malignant(X) :- lesion_kind(X, mass), shape(X, irregular).
malignant(X) :- lesion_kind(X, mass), margin(X, spiculated).
malignant(X) :- lesion_kind(X, calcification), calc(X, suspicious).
'''.strip()
    PROGRAM_PATH.parent.mkdir(parents=True, exist_ok=True)
    PROGRAM_PATH.write_text(PROGRAM + "\n")
    print("Program:", PROGRAM_PATH)
    print(PROGRAM)

    sw = Network(shape_net, "shape_net", batching=True)
    mw = Network(margin_net, "margin_net", batching=True)
    cw = Network(calc_net, "calc_net", batching=True)
    kw = Network(kind_net, "kind_net", batching=True)
    sw.optimizer = torch.optim.Adam(shape_net.parameters(), lr=LR)
    mw.optimizer = torch.optim.Adam(margin_net.parameters(), lr=LR)
    cw.optimizer = torch.optim.Adam(calc_net.parameters(), lr=LR)
    kw.optimizer = torch.optim.Adam(kind_net.parameters(), lr=LR)

    model = Model(str(PROGRAM_PATH), [sw, mw, cw, kw])
    model.set_engine(ExactEngine(model), cache=False)
    model.add_tensor_source("train", TensorSource(train_tensors))
    model.add_tensor_source("val", TensorSource(val_tensors))
    model.add_tensor_source("test", TensorSource(test_tensors))

    loader = DataLoader(train_queries, BATCH_SIZE, shuffle=True, seed=RANDOM_SEED)
    print(f"Smoke-test DeepProbLog: n_queries={len(train_queries)}")
    train_model(model, loader, EPOCHS, log_iter=max(1, len(loader)))
    print("Training finished.")


## Cell: metrics (including F1)

**What:** for validation and test, query `malignant(X)`, threshold at 0.5, and report accuracy, precision, sensitivity, specificity, F1 for malignant, F1 for benign, and macro F1.

**Why:** accuracy alone can look fine if the model always answers the majority class. If that happens, one of the F1 scores goes to 0 and `pct_pred_malignant` sits near 0 or 1. This smoke test should show **both** classes being predicted, and `P(malignant)` should differ across images.


In [ ]:
if not RUN_DEEPPROBLOG:
    print("No metrics: DeepProbLog was skipped.")
else:
    def extract_prob(result):
        mapping = result.result if hasattr(result, "result") else {}
        if not mapping:
            return 0.0
        vals = [float(v.detach().cpu().item()) if hasattr(v, "detach") else float(v) for v in mapping.values()]
        return sum(vals) / len(vals)

    def _f1(tp, fp, fn):
        prec = tp / max(tp + fp, 1)
        rec = tp / max(tp + fn, 1)
        return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

    def eval_split(name, dset, df):
        y_true, y_prob = [], []
        for i in range(len(dset)):
            y_true.append(int(df.iloc[i]["malignant"]))
            q = Query(Term("malignant", Term("tensor", Term(name, Constant(i)))))
            y_prob.append(extract_prob(model.solve([q])[0]))
        y_true, y_prob = np.array(y_true), np.array(y_prob)
        y_pred = (y_prob >= 0.5).astype(int)
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        tn = int(((y_pred == 0) & (y_true == 0)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())
        f1_mal, f1_ben = _f1(tp, fp, fn), _f1(tn, fn, fp)
        return {
            "split": name, "n": len(y_true),
            "accuracy": (tp + tn) / max(len(y_true), 1),
            "precision": tp / max(tp + fp, 1),
            "sensitivity": tp / max(tp + fn, 1),
            "specificity": tn / max(tn + fp, 1),
            "f1_malignant": f1_mal,
            "f1_benign": f1_ben,
            "f1_macro": 0.5 * (f1_mal + f1_ben),
            "tp": tp, "tn": tn, "fp": fp, "fn": fn,
            "pct_pred_malignant": float(y_pred.mean()),
        }, y_prob, y_pred

    metrics_val, val_prob, val_pred = eval_split("val", val_set, df_val)
    metrics_test, test_prob, test_pred = eval_split("test", test_set, df_test)
    metrics_df = pd.DataFrame([metrics_val, metrics_test])
    display(metrics_df)
    print("If the model predicts only one class: one F1 goes to 0 and pct_pred_malignant is near 0 or 1.")


## Cell: show test images with predictions

**What:** draw the **test** placeholder images again. Each title is:

`patient | gold pathology | P(malignant) | pred`

Green title = prediction matches the gold label. Red title = mismatch.

**Why:** this is the visual check for the smoke test. You should see some benign images with low `P(malignant)` and some malignant images with high `P(malignant)`, not the same probability on every picture.


In [ ]:
if not RUN_DEEPPROBLOG:
    print("No prediction grid: DeepProbLog was skipped.")
else:
    def show_predictions(title, df, tensors, y_prob, y_pred, n=SHOW_N_IMAGES):
        n = min(int(n), len(df))
        cols = 4
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(13, 3.1 * rows))
        axes = np.atleast_1d(axes).ravel()
        for i in range(n):
            ax = axes[i]
            img = tensors[i].squeeze().detach().cpu().numpy()
            ax.imshow(img, cmap="gray", vmin=0, vmax=1)
            row = df.iloc[i]
            gold = "MAL" if int(row["malignant"]) == 1 else "BEN"
            pred = "MAL" if int(y_pred[i]) == 1 else "BEN"
            ok = gold == pred
            ax.set_title(
                f"{row['patient_id']}  gold={gold} pred={pred}\nP={y_prob[i]:.2f}  {row['abnormality_kind']}",
                fontsize=8,
                color=("green" if ok else "red"),
            )
            ax.axis("off")
        for j in range(n, len(axes)):
            axes[j].axis("off")
        fig.suptitle(title)
        fig.tight_layout()
        plt.show()

    show_predictions("Validation images + predictions", df_val, val_tensors, val_prob, val_pred)
    show_predictions("Test images + predictions", df_test, test_tensors, test_prob, test_pred)


## Cell: reasoning traces + save

**What:** print a few validation cases with the concept probabilities the rules actually use. Then save the model weights and the metrics JSON.

**Why:** if `P(malignant)` is high, this cell shows *which* concept fired (kind, irregular, spiculated, or suspicious calc). That is how you debug a wrong call in the smoke test.


In [ ]:
if not RUN_DEEPPROBLOG:
    print("No traces: DeepProbLog was skipped.")
else:
    print("Example traces (validation):")
    for i in range(min(3, len(df_val))):
        row = df_val.iloc[i]
        img = Term("tensor", Term("val", Constant(i)))
        print(
            f"[{i}] pathology={row['pathology']} gold={row['shape_concept']}/{row['margin_concept']}/{row['calc_concept']} kind={row['kind_concept']}"
        )
        for name, q in {
            "malignant": Query(Term("malignant", img)),
            "kind_mass": Query(Term("lesion_kind", img, Term("mass"))),
            "kind_calc": Query(Term("lesion_kind", img, Term("calcification"))),
            "shape_irregular": Query(Term("shape", img, Term("irregular"))),
            "margin_spiculated": Query(Term("margin", img, Term("spiculated"))),
            "calc_suspicious": Query(Term("calc", img, Term("suspicious"))),
        }.items():
            print(f"  P({name}) = {extract_prob(model.solve([q])[0]):.3f}")

    snap = OUT_DIR / "deepproblog_local_smoke.pth"
    model.save_state(str(snap))
    (OUT_DIR / "deepproblog_local_metrics.json").write_text(
        json.dumps({"smoke": True, "metrics": [metrics_val, metrics_test]}, indent=2)
    )
    print("Saved", snap)
    print("LOCAL SMOKE OK — this is a wiring test on a small sample.")
